In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/pss5e10-main/__results__.html
/kaggle/input/pss5e10-main/xgb_5_fold_te_log_transformed.csv
/kaggle/input/pss5e10-main/__notebook__.ipynb
/kaggle/input/pss5e10-main/__output__.json
/kaggle/input/pss5e10-main/Train_org.csv
/kaggle/input/pss5e10-main/test.csv
/kaggle/input/pss5e10-main/final_train.csv
/kaggle/input/pss5e10-main/custom.css
/kaggle/input/pss5e10-main/__results___files/__results___11_2.png
/kaggle/input/playground-series-s5e10/sample_submission.csv
/kaggle/input/playground-series-s5e10/train.csv
/kaggle/input/playground-series-s5e10/test.csv


In [2]:
!pip install autogluon.tabular[0]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.3/487.3 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.0/278.0 kB 17.3 MB/s eta 0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 7.1.0
    Uninstalling psutil-7.1.0:
      Successfully uninstalled psutil-7.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.

In [3]:
train = pd.read_csv('/kaggle/input/pss5e10-main/final_train.csv')
# train = train.fillna(0)
test = pd.read_csv('/kaggle/input/pss5e10-main/test.csv')
test = test.drop(columns='accident_risk')
train['residual_risk'] = train['accident_risk'] - train['y']
train.drop(columns='accident_risk', inplace=True)

In [4]:
TARGET = 'residual_risk'
FEATURES = [col for col in train.columns if col!='residual_risk']

In [5]:
train = train.fillna(0)
test = test.fillna(0)

In [6]:
import shutil, os
old = '/kaggle/working/AutogluonModels'
if os.path.exists(old):
    shutil.rmtree(old)

In [7]:
from autogluon.tabular import TabularPredictor
import warnings
warnings.filterwarnings('ignore')

PEAK_XGB = {
    'n_estimators': 100_000, 'learning_rate': 0.01, 'max_depth': 6,
    'subsample': 0.9, 'colsample_bytree': 0.6, 'reg_alpha': 0.0, 'reg_lambda': 0.0,
    'tree_method': 'gpu_hist', 'device': 'cuda', 'n_jobs': -1, 'verbosity': 0
}

PEAK_LGB = {
    'n_estimators': 100_000, 'learning_rate': 0.01, 'num_leaves': 64,
    'subsample': 0.9, 'colsample_bytree': 0.6, 'reg_alpha': 0.0, 'reg_lambda': 0.0,
    'device': 'gpu', 'n_jobs': -1, 'verbosity': -1
}

PEAK_CAT = {
    'iterations': 100_000, 'learning_rate': 0.01, 'depth': 6,
    'l2_leaf_reg': 0.0, 'subsample': 0.9, 'task_type': 'GPU', 'verbose': False
}

# ----------  AutoGluon search space  ----------
predictor = TabularPredictor(
    label=TARGET,
    eval_metric='rmse',
    problem_type='regression',
    path='AutogluonModels/peak_locked'
).fit(
    train_data=train,
    time_limit=12000,                     # 15 min total
    presets='best_quality',
    num_bag_folds=7,                    # same CV you trust
    num_bag_sets=3,                     # 3×7 OOF bags
    num_stack_levels=3,                 # 2-level stacking
    raise_on_no_models_fitted=False,
    hyperparameters={
        # ----  your peak configs (locked)  ----
        'XGB': [PEAK_XGB],               # list → AutoGluon **will not tune inside**
        'GBM': [PEAK_LGB],
        'CAT': [PEAK_CAT],

        # ----  diversity engines (small search)  ----
        'NN_TORCH': [
            {'num_layers': 2, 'hidden_size': 128, 'dropout': 0.1},
            {'num_layers': 3, 'hidden_size': 256, 'dropout': 0.2}
        ],
        # 'RF':  {'n_estimators': 300, 'max_depth': 8, 'n_jobs': -1},
        # 'XT':  {'n_estimators': 300, 'max_depth': 8, 'n_jobs': -1},
        # 'LR':  {},
    },
    # hyperparameter_tune_kwargs={
    #     'num_trials': 10,               # only for NN / RF / XT
    #     'searcher': 'auto',
    #     'time_limit_per_model': 30,     # seconds per trial
    # },
    verbosity=1,
    num_gpus=1
)

Will use sequential fold fitting strategy because import of ray failed. Reason: ray==2.49.2 detected. 2.10.0 <= ray < 2.45.0 is required. You can use pip to install certain version of ray `pip install "ray>=2.10.0,<2.45.0"`
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
		EmbedNet._set_params() got an unexpected keyword argument 'dropout'
Detailed Trac

In [8]:
leaderboard = predictor.leaderboard()
leaderboard

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L5,-0.055901,root_mean_squared_error,337.193352,3789.689123,0.006953,0.421967,5,True,12
1,WeightedEnsemble_L3,-0.055907,root_mean_squared_error,259.446821,2705.975415,0.007101,0.136613,3,True,6
2,LightGBM_BAG_L2,-0.055908,root_mean_squared_error,257.087070,2540.987009,31.760795,329.521458,2,True,4
3,XGBoost_BAG_L2,-0.055913,root_mean_squared_error,227.678925,2376.317344,2.352650,164.851794,2,True,5
4,LightGBM_BAG_L4,-0.055914,root_mean_squared_error,334.721293,3589.521366,47.249076,427.316104,4,True,10
5,XGBoost_BAG_L4,-0.055916,root_mean_squared_error,289.937323,3361.951052,2.465106,199.745791,4,True,11
6,LightGBM_BAG_L3,-0.055916,root_mean_squared_error,285.178982,3008.621258,25.739262,302.782455,3,True,7
7,WeightedEnsemble_L4,-0.055916,root_mean_squared_error,285.186196,3008.759627,0.007214,0.138369,4,True,9
8,XGBoost_BAG_L3,-0.055922,root_mean_squared_error,261.732956,2859.422806,2.293236,153.584004,3,True,8
9,WeightedEnsemble_L2,-0.055929,root_mean_squared_error,225.333393,2211.600332,0.007118,0.134782,2,True,3


In [9]:
models = leaderboard['model'].to_list()
oofs_dict = {}
for m in models:
    oofs_dict[m] = predictor.predict_oof(model=m)

oofs_df = pd.DataFrame(oofs_dict)

In [10]:
predictor2 = TabularPredictor.load('/kaggle/working/AutogluonModels/peak_locked')
test_preds = {}
for m in models:
    test_preds[m] = predictor2.predict(test, model=m)

test_preds_df = pd.DataFrame(test_preds)

In [11]:
for col in oofs_df.columns:
    oofs_df[col] = oofs_df[col] + train['y']
    test_preds_df[col] = test_preds_df[col] + test['y']

In [12]:
samp = pd.read_csv('/kaggle/input/playground-series-s5e10/sample_submission.csv')
samp['accident_risk'] = test_preds_df.iloc[:,0]
samp.to_csv('sample_submission.csv', index=False)

In [13]:
leaderboard.to_csv('leaderboard_autogluon_residuals.csv', index=False)
oofs_df.to_csv('oofs_autogluon_residuals.csv', index=False)
test_preds_df.to_csv('test_preds_autogluon_residuals.csv', index=False)